# 7.6 Get Your Survey Data Machine Learning Ready — Code Brief

## Key Concepts

- **Feature fusion**: combine structured demographics/academics with compressed text features into one master matrix.
- Structured features scaled via `ColumnTransformer`: MinMaxScaler (academic ratios), StandardScaler (unit counts), OneHotEncoder (demographics).
- **PCA** reduces the hundreds of TF-IDF text columns down to a handful of components (80% cumulative explained variance threshold) so text doesn't mathematically drown out the structured features.
- PCA is fit on training data only, then `.transform()` (not `.fit_transform()`) on test data — avoids leakage.
- Final output: `ML_SURVEY_MASTER_TRAIN.csv` / `ML_SURVEY_MASTER_TEST.csv`, ready for downstream modeling.

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(42)
random.seed(42)

filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
df_training = pd.read_csv(f'{filepath}training.csv')
df_testing = pd.read_csv(f'{filepath}testing.csv')


print("Training size:", len(df_training))
print("Test size:", len(df_testing))


## Preprocess Structured Student Data

In [ ]:
minmax_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2']
standard_cols = ['UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
categorical_cols = ['GENDER', 'RACE_ETHNICITY', 'FIRST_GEN_STATUS']

# Drop rows missing any of these key columns
df_train = df_training.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()
df_test = df_testing.dropna(subset=minmax_cols + standard_cols + categorical_cols).copy()

print(f"Training rows after dropping incomplete records: {len(df_train)}")

preprocessor = ColumnTransformer(
    transformers=[
        ('minmax',   MinMaxScaler(), minmax_cols),
        ('standard', StandardScaler(), standard_cols),
        ('onehot',   OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ],
    remainder='drop'
)

X_structured_train = preprocessor.fit_transform(df_train)
df_structured_train = pd.DataFrame(X_structured_train, index=df_train.index)

X_structured_test = preprocessor.transform(df_test)
df_structured_test = pd.DataFrame(X_structured_test, index=df_test.index)

print("Structured feature matrix:", df_structured_train.shape)


## Reduce TF-IDF Text Features with PCA

In [ ]:
ML_Survey_Data_Num = pd.read_csv(f'{filepath}ML_Survey_Data_Num.csv')
ML_Survey_Data22_Num = pd.read_csv(f'{filepath}ML_Survey_Data22_Num.csv')
display(ML_Survey_Data_Num)

In [ ]:
tfidf_matrix_train = ML_Survey_Data_Num.iloc[:,11:]
print("TF-IDF matrix:", tfidf_matrix_train.shape)
tfidf_matrix_train

In [ ]:
tfidf_matrix_test = ML_Survey_Data22_Num.iloc[:,11:]
print("TF-IDF matrix:", tfidf_matrix_test.shape)


In [ ]:
# Choose number of PCA components by explained variance
pca_full = PCA(random_state=42)
pca_full.fit(tfidf_matrix_train)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
threshold = 0.80
n_components = int(np.searchsorted(cumvar, threshold)) + 1

fig = px.line(
    x=range(1, len(cumvar) + 1), y=cumvar,
    labels={'x': 'Number of PCA Components', 'y': 'Cumulative Explained Variance'},
    title='PCA Explained Variance — TF-IDF Features'
)
fig.add_hline(y=threshold, line_dash='dash', annotation_text=f'{int(threshold*100)}% threshold')
fig.add_vline(x=n_components, line_dash='dot',
              annotation_text=f'{n_components} components', annotation_position='top right')
fig.show()
print(f"→ Using {n_components} PCA components to capture {threshold*100:.0f}% of text variance")

In [ ]:
# Apply PCA with the chosen number of components
pca = PCA(n_components=n_components, random_state=42)
X_text_pca_train = pca.fit_transform(tfidf_matrix_train)
df_text_pca_train = pd.DataFrame(X_text_pca_train, index=df_train.index,
                           columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca_train.shape)


In [ ]:
# Apply PCA with the chosen number of components
X_text_pca_test = pca.transform(tfidf_matrix_test)
df_text_pca_test = pd.DataFrame(X_text_pca_test, index=df_test.index,
                           columns=[f'TEXT_PC{i+1}' for i in range(n_components)])
print("Text PCA matrix:", df_text_pca_test.shape)

## Feature Fusion: Creating the Master Matrix

In [ ]:
df_all_train = pd.concat([df_structured_train, df_text_pca_train], axis=1)
df_all_train.columns = df_all_train.columns.astype(str)  # KMeans requires string column names
print("Combined feature matrix:", df_all_train.shape)
df_all_train

In [ ]:
df_all_test = pd.concat([df_structured_test, df_text_pca_test], axis=1)
df_all_test.columns = df_all_test.columns.astype(str)  # KMeans requires string column names
print("Combined feature matrix:", df_all_test.shape)
df_all_test

In [ ]:
transformed_structured_feature_names = preprocessor.get_feature_names_out()

cleaned_structured_cols = []
for col_name in transformed_structured_feature_names:
    if col_name.startswith('minmax__'):
        cleaned_structured_cols.append(col_name.replace('minmax__', ''))
    elif col_name.startswith('standard__'):
        cleaned_structured_cols.append(col_name.replace('standard__', ''))
    elif col_name.startswith('onehot__'):
        # Remove the 'onehot__' prefix to make names like 'GENDER_Female' cleaner
        cleaned_structured_cols.append(col_name.replace('onehot__', ''))
    else:
        cleaned_structured_cols.append(col_name)

# The PCA column names are already correctly named in df_text_pca_train.columns
pca_cols = df_text_pca_train.columns.tolist()

# Combine all column names
df_all_train.columns = cleaned_structured_cols + pca_cols
df_all_test.columns = cleaned_structured_cols + pca_cols


print("Combined feature matrix with renamed columns:", df_all_train.shape)
display(df_all_train.head(3))

#print("Combined test feature matrix with renamed columns:", df_all_test.shape)
#display(df_all_test.head(3))

ML_Survey_Data_train = df_all_train
ML_Survey_Data_test = df_all_test

ML_Survey_Data_train['SEM_3_STATUS'] = df_training['SEM_3_STATUS']
ML_Survey_Data_test['SEM_3_STATUS'] = df_testing['SEM_3_STATUS']

ML_Survey_Data_train


In [ ]:
save_path = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

train = ML_Survey_Data_train
test = ML_Survey_Data_test

train.to_csv(save_path + 'ML_SURVEY_MASTER_TRAIN.csv', index=False)
test.to_csv(save_path + 'ML_SURVEY_MASTER_TEST.csv', index=False)